## 연습 1: Northbank Financial 데이터 모델 만들기

이 연습에서는 Northbank Financial의 분석 플랫폼을 위한 기초 Delta 테이블을 설계합니다. 이 모델에는 SCD Type 2 이력 추적용으로 설계된 **고객 차원(customer dimension)**과 liquid clustering 및 Change Data Feed로 최적화된 **트랜잭션 팩트 테이블(transaction fact table)**이 포함됩니다.

이 연습을 완료한 후에는 랩 설정 페이지로 돌아가 **Catalog Explorer**에서 만든 테이블을 살펴보세요.

### 작업 1.1 — 카탈로그 및 스키마 만들기

`banking_lab`이라는 이름의 Unity Catalog **카탈로그**를 적절한 주석과 함께 만듭니다. 그 안에 `silver`라는 이름의 **스키마**를 만듭니다.

In [ ]:
# 기본 스토리지가 활성화되어 있지 않으므로, 기본 카탈로그의 루트에서 스토리지 경로를 상속받습니다.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"스토리지 루트: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS banking_lab
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Northbank Financial 레이크하우스 플랫폼 — 고객 분석 및 규제 보고용'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS banking_lab.silver
  COMMENT '정제 및 보강된 뱅킹 데이터 — 고객 차원 및 트랜잭션 팩트';

### 작업 1.2 — `dim_customer` SCD Type 2 테이블 만들기

관리형 Delta 테이블 `banking_lab.silver.dim_customer`를 만듭니다. 이 테이블은 SCD Type 2 패턴을 사용하여 **고객 차원 이력**을 추적합니다.

테이블에는 다음 항목이 포함되어야 합니다:

| Column | Type | Notes |
|---|---|---|
| `customer_sk` | `BIGINT GENERATED ALWAYS AS IDENTITY` | 대리 키(surrogate key) — 자동 증가하며 버전마다 고유 |
| `customer_id` | `STRING NOT NULL` | 원본 시스템의 비즈니스/자연 키 |
| `full_name` | `STRING` | |
| `email` | `STRING` | |
| `city` | `STRING` | 주 거주 도시 |
| `segment` | `STRING` | 고객 세그먼트: Retail, Premium, Business |
| `account_type` | `STRING` | 계정 유형: Current, Savings, Business |
| `valid_from` | `TIMESTAMP NOT NULL` | 이 버전이 적용되기 시작한 시점 |
| `valid_to` | `TIMESTAMP NOT NULL` | 이 버전이 만료된 시점 (활성 레코드는 9999-12-31) |
| `is_current` | `BOOLEAN` | 활성 버전이면 True |

`customer_id`에 **liquid clustering**을 적용하고 **Change Data Feed**를 활성화합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"TBLPROPERTIES에서 GENERATED ALWAYS AS IDENTITY, CLUSTER BY, Change Data Feed를 활성화하여 Databricks에서 Delta 테이블을 어떻게 만드나요?"*

**힌트:**
- 조회 성능을 최적화하려면 `CLUSTER BY (customer_id)`를 사용하세요
- 감사 추적 지원을 위해 `TBLPROPERTIES (delta.enableChangeDataFeed = true)`를 추가하세요
- `USING DELTA`를 사용하세요 (Azure Databricks의 모든 테이블에 권장되는 형식입니다)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.dim_customer (
    customer_sk  BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id  STRING NOT NULL,
    full_name    STRING,
    email        STRING,
    city         STRING,
    segment      STRING,
    account_type STRING,
    valid_from   TIMESTAMP NOT NULL,
    valid_to     TIMESTAMP NOT NULL,
    is_current   BOOLEAN
)
USING DELTA
CLUSTER BY (customer_id)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);

### 작업 1.3 — `fact_transactions` 테이블 만들기

개별 뱅킹 결제를 저장할 관리형 Delta 테이블 `banking_lab.silver.fact_transactions`를 만듭니다.

| Column | Type | Notes |
|---|---|---|
| `transaction_id` | `STRING NOT NULL` | 고유 트랜잭션 참조 |
| `account_id` | `STRING` | 은행 계좌 식별자 |
| `customer_id` | `STRING` | `dim_customer`에 연결 |
| `transaction_type` | `STRING` | Credit, Debit 또는 Transfer |
| `amount` | `DECIMAL(15,2)` | 트랜잭션 금액 |
| `currency` | `STRING DEFAULT 'GBP'` | 통화 코드 |
| `transaction_date` | `DATE` | 트랜잭션 날짜 |
| `description` | `STRING` | 트랜잭션 내역 |

예상되는 쿼리 패턴(고객 및/또는 날짜 범위로 필터링)을 지원하도록 `customer_id, transaction_date`에 **liquid clustering**을 적용합니다. FCA 규정 준수 감사를 위해 **Change Data Feed**를 활성화합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks SQL에서 liquid clustering과 Change Data Feed를 활성화하여 Delta Lake 테이블을 어떻게 만드나요?"*

**힌트:**
- `CLUSTER BY (customer_id, transaction_date)`를 사용하세요 — 클러스터링 키는 최대 4개까지 지정할 수 있습니다
- `TBLPROPERTIES (delta.enableChangeDataFeed = true)`를 추가하세요
- 통화 컬럼의 기본값으로 `STRING DEFAULT 'GBP'`를 사용하세요

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS banking_lab.silver.fact_transactions (
    transaction_id   STRING NOT NULL,
    account_id       STRING,
    customer_id      STRING,
    transaction_type STRING,
    amount           DECIMAL(15,2),
    currency         STRING DEFAULT 'GBP',
    transaction_date DATE,
    description      STRING
)
USING DELTA
CLUSTER BY (customer_id, transaction_date)
TBLPROPERTIES (
    delta.enableChangeDataFeed = true,
    delta.feature.allowColumnDefaults = 'supported'
);

## 연습 2: 고객 차원에 SCD Type 2 구현하기

Northbank의 규제 보고는 현재 상태만이 아니라 **각 트랜잭션 시점의 고객 프로필**을 반영해야 합니다. 이 연습에서는 초기 고객 레코드를 적재하고, 현실적인 속성 변경을 적용하며, SCD Type 2를 위한 **2단계 MERGE + INSERT 패턴**을 구현합니다. 그런 다음 전체 고객 이력을 조회하고 특정 시점 조회(point-in-time lookup)를 수행합니다.

### 작업 2.1 — 초기 고객 레코드 적재하기

아래 셀은 8명의 Northbank 고객에 대한 **초기 데이터셋**을 PySpark DataFrame으로 정의합니다. 여러분의 작업은 이 DataFrame을 append 모드로 **`banking_lab.silver.dim_customer`에 기록**하는 것입니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 saveAsTable을 append 모드로 사용하여 PySpark DataFrame을 기존 Delta 테이블에 어떻게 기록하나요?"*

**힌트:**
- `df.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")`를 사용하세요
- 데이터와 DataFrame 설정은 이미 제공되어 있습니다 — DataFrame이 빌드된 후에 write 문을 추가하세요

In [ ]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

df_initial.write.mode("append").saveAsTable("banking_lab.silver.dim_customer")
print(f"{df_initial.count()}건의 초기 고객 레코드를 적재했습니다.")

In [ ]:
%sql
-- 확인: 8건의 고객 레코드가 표시되어야 합니다
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM banking_lab.silver.dim_customer
ORDER BY customer_sk;

### 작업 2.2 — 고객 변경 사항이 포함된 스테이징 데이터 적재하기

운영팀이 다음과 같은 변경 알림을 받았습니다:

| Customer | Change |
|---|---|
| **C-1001 Emma Hartley** | **London → Oxford**로 이사, **Premium** 세그먼트로 업그레이드 |
| **C-1003 Sophia Chen** | **Edinburgh → London**으로 이전 |
| **C-1006 Liam Murray** | 이메일 주소만 변경 (도시와 세그먼트는 변동 없음) |
| **C-1009 Priya Singh** | Cardiff에서 합류하는 **신규 고객**, Retail 세그먼트 |

아래 셀을 실행하여 스테이징 DataFrame을 만들고 임시 뷰로 등록하세요. 다음 작업에서 사용합니다.

In [ ]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("스테이징 데이터를 적재했습니다. 임시 뷰 'staging_customers'를 사용할 수 있습니다.")
display(df_staging)

### 작업 2.3 — SCD Type 2 적용: 1단계 — 변경된 레코드 닫기

`staging_customers`를 `dim_customer`의 현재 버전과 비교하는 `MERGE INTO` 문을 작성합니다. `city`, `segment`, `email` 또는 `account_type`이 변경된 레코드에 대해서는 다음 값을 설정하여 **현재 버전을 닫습니다**:
- `valid_to = current_timestamp()`
- `is_current = false`

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"추적 대상 속성이 변경되면 is_current=false 및 valid_to=current_timestamp()로 설정하여 SCD Type 2 차원 레코드를 닫는 Databricks SQL MERGE 문을 작성해 주세요."*

**힌트:**
- `target.customer_id = source.customer_id AND target.is_current = true`로 매칭하세요
- `WHEN MATCHED AND (...)` 조건은 추적 대상 속성 중 **하나라도** 다른지 확인해야 합니다
- `valid_to`와 `is_current`만 업데이트하세요 — 이력 데이터 컬럼은 덮어쓰지 마세요

In [ ]:
%sql
-- 1단계: 변경된 고객 레코드의 현재 버전 닫기
MERGE INTO banking_lab.silver.dim_customer AS target
USING staging_customers AS source
ON target.customer_id = source.customer_id
   AND target.is_current = true
WHEN MATCHED AND (
    target.city         <> source.city         OR
    target.segment      <> source.segment       OR
    target.email        <> source.email         OR
    target.account_type <> source.account_type
) THEN UPDATE SET
    target.valid_to    = current_timestamp(),
    target.is_current  = false;

### 작업 2.3 — SCD Type 2 적용: 2단계 — 새 버전 및 신규 고객 삽입

다음을 수행하는 `INSERT INTO` 문을 작성합니다:
1. 방금 이전 버전이 닫힌 고객(C-1001, C-1003, C-1006)에 대해 **새로운 현재 행(current row)** 추가
2. 기존 레코드가 없는 **완전히 새로운 고객** C-1009 추가

새 행은 `valid_from = current_timestamp()`, `valid_to = TIMESTAMP '9999-12-31'`, `is_current = true`를 가져야 합니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"is_current=true 레코드가 존재하지 않는 고객에 대해서만, 스테이징 테이블에서 새로운 현재 SCD Type 2 행을 추가하는 Databricks SQL INSERT 문을 작성해 주세요."*

**힌트:** `WHERE NOT EXISTS (SELECT 1 FROM banking_lab.silver.dim_customer d WHERE d.customer_id = s.customer_id AND d.is_current = true)`를 사용하여 어떤 스테이징 레코드에 새 행을 삽입해야 하는지 식별하세요.

In [ ]:
%sql
-- 2단계: 변경된 고객 및 신규 고객의 새 현재 버전 삽입
INSERT INTO banking_lab.silver.dim_customer
    (customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current)
SELECT
    s.customer_id,
    s.full_name,
    s.email,
    s.city,
    s.segment,
    s.account_type,
    current_timestamp()              AS valid_from,
    TIMESTAMP '9999-12-31 00:00:00'  AS valid_to,
    true                             AS is_current
FROM staging_customers s
WHERE NOT EXISTS (
    SELECT 1
    FROM banking_lab.silver.dim_customer d
    WHERE d.customer_id = s.customer_id
      AND d.is_current = true
);

### 작업 2.4 — 고객의 전체 버전 이력 조회하기

고객 `C-1001`(Emma Hartley)의 **모든 버전**을 `valid_from` 순으로 정렬하여 표시하는 쿼리를 작성합니다. 다음과 같이 표시되어야 합니다:
- 버전 1: London / Retail (만료됨)
- 버전 2: Oxford / Premium (현재)

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks Delta 테이블에서 특정 고객의 모든 SCD Type 2 버전을 어떻게 조회하나요?"*

**힌트:** `customer_id = 'C-1001'`로 필터링하고 `ORDER BY valid_from ASC`를 사용하세요.

In [ ]:
%sql
-- 고객 C-1001의 모든 버전을 valid_from 순으로 표시
SELECT customer_sk, customer_id, full_name, city, segment, valid_from, valid_to, is_current
FROM banking_lab.silver.dim_customer
WHERE customer_id = 'C-1001'
ORDER BY valid_from ASC;

### 작업 2.5 — 특정 시점 고객 조회

Northbank의 규제 보고팀은 위의 변경 사항이 적용되기 전인 **2020-06-15 시점에 존재했던 고객 프로필**을 알아야 합니다.

해당 날짜의 고객별 프로필을 한 행씩 반환하는 쿼리를 작성합니다. `valid_from`과 `valid_to` 컬럼을 사용하여 2020-06-15에 활성 상태였던 버전을 선택하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"valid_from과 valid_to 날짜 범위 필터링을 사용하여 SCD Type 2 테이블에서 특정 시점 쿼리를 어떻게 작성하나요?"*

**힌트:** 날짜 `D`에 활성 상태였던 버전은 `valid_from <= D AND valid_to > D`인 경우입니다. 예상 결과: 8명의 모든 원본 고객이 초기 상태로 표시됩니다(Emma는 Oxford 아님, Sophia는 London 아님).

In [ ]:
%sql
-- 2020-06-15 시점에 존재했던 고객 프로필 반환
SELECT customer_id, full_name, city, segment, account_type
FROM banking_lab.silver.dim_customer
WHERE valid_from <= TIMESTAMP '2020-06-15 00:00:00'
  AND valid_to   >  TIMESTAMP '2020-06-15 00:00:00'
ORDER BY customer_id;

## 연습 3: Change Data Feed & FCA 감사 추적

Basel III 및 FCA 규정은 Northbank가 트랜잭션 데이터에 대한 모든 수정 사항의 **조회 가능한 감사 추적**을 유지하도록 요구합니다. 이 연습에서는 결제 트랜잭션을 적재하고, (실제 현장에서 흔히 발생하는) 데이터 정정을 시뮬레이션한 다음, **Change Data Feed**를 조회하여 전체 감사 로그를 생성합니다.

### 작업 3.1 — 초기 트랜잭션 적재하기

아래 셀을 실행하여 10건의 결제 트랜잭션을 `fact_transactions`에 삽입합니다. 이는 하루치 배치로 처리된 결제를 나타냅니다.

In [ ]:
%sql
INSERT INTO banking_lab.silver.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

### 작업 3.2 — 트랜잭션 정정 시뮬레이션

Northbank의 정산팀이 두 건의 금액 불일치를 발견했습니다. 아래 셀을 실행하여 정정을 적용하세요.

In [ ]:
%sql
-- 정정 1: TXN-003 금액이 잘못 입력됨 (정확한 값: £275.50, £250.00 아님)
UPDATE banking_lab.silver.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

-- 정정 2: TXN-007 금액은 분쟁 중
UPDATE banking_lab.silver.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

### 작업 3.3 — Change Data Feed를 조회하여 감사 추적 구성하기

`table_changes()`를 사용하여 `fact_transactions`의 이력 시작 시점 이후 기록된 **모든 변경 이벤트**를 가져오는 쿼리를 작성합니다. `_change_type`, `_commit_version`, `_commit_timestamp` 메타데이터 컬럼을 포함하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 table_changes()를 사용하여 Delta 테이블의 Change Data Feed를 어떻게 조회하나요? 변경 유형과 커밋 메타데이터 컬럼을 보여주세요."*

**힌트:** `table_changes('banking_lab.silver.fact_transactions', 0)`을 사용하세요 — 두 번째 인수는 시작 버전 번호입니다.

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_version,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
ORDER BY _commit_timestamp, transaction_id;

### 작업 3.4 — FCA 보고를 위해 정정된 레코드 분리하기

FCA 보고를 위해 규정 준수팀은 수정된 트랜잭션의 **정정 후 상태**만 보여주는 뷰가 필요합니다. Change Data Feed를 필터링하여 `update_postimage` 레코드만 표시하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 table_changes() 결과를 필터링하여 update_postimage 레코드만 표시하려면 어떻게 하나요?"*

**힌트:** 각 `UPDATE` 작업은 두 개의 행을 생성합니다: `update_preimage`(변경 전)와 `update_postimage`(변경 후). `_change_type = 'update_postimage'`로 필터링하세요. 예상 결과: TXN-003은 £275.50, TXN-007은 £1450.00.

In [ ]:
%sql
SELECT
    transaction_id,
    amount,
    description,
    _change_type,
    _commit_timestamp
FROM table_changes('banking_lab.silver.fact_transactions', 0)
WHERE _change_type = 'update_postimage'
ORDER BY _commit_timestamp;

## 연습 4: Delta Lake Time Travel

Delta Lake는 테이블에 적용된 모든 변경 사항의 전체 **트랜잭션 로그**를 유지합니다. 이를 통해 데이터의 **이전 버전**을 조회하고, 백업에서 복원하지 않고도 테이블을 이전 상태로 **복원(restore)**할 수 있습니다. 이는 트랜잭션 데이터가 실수로 수정되었을 때 Northbank에게 매우 중요합니다.

### 작업 4.1 — 트랜잭션 이력 살펴보기

`DESCRIBE HISTORY`를 사용하여 `fact_transactions`의 전체 작업 로그를 확인합니다. 출력에는 작업 유형, 타임스탬프, 사용자 등 테이블의 모든 버전이 표시됩니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 모든 버전 번호를 포함하여 Delta 테이블의 전체 작업 이력을 어떻게 확인하나요?"*

출력의 **버전 번호**를 기록해 두세요 — 다음 작업에서 사용합니다.

In [ ]:
%sql
DESCRIBE HISTORY banking_lab.silver.fact_transactions;

### 작업 4.2 — 정정 전 원래 트랜잭션 금액 조회하기

`VERSION AS OF`를 사용하여 정정이 적용되기 전, 즉 **초기 INSERT 직후 시점의** `TXN-003`과 `TXN-007`을 가져옵니다.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 SELECT 문에 VERSION AS OF를 사용하여 Delta 테이블의 이전 버전을 어떻게 조회하나요?"*

**힌트:** 초기 INSERT는 버전 **1**에 해당합니다(버전 0은 테이블 생성). 예상 결과: TXN-003은 £250.00, TXN-007은 £1500.00.

In [ ]:
%sql
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions VERSION AS OF 1
WHERE transaction_id IN ('TXN-003', 'TXN-007');

### 작업 4.3 — 테이블을 정정 전 상태로 복원하기

정산팀이 정정이 잘못 적용되었음을 확인했으며 이를 롤백해야 합니다. `RESTORE TABLE`을 사용하여 `fact_transactions`를 **버전 1**(초기 적재 후, 정정 전)로 되돌립니다.

복원한 후 TXN-003과 TXN-007이 원래 금액을 표시하는지 확인하세요.

> 🤖 **Genie Code tip:** 질문해 보세요:
> *"Databricks에서 RESTORE TABLE을 사용하여 Delta 테이블을 특정 버전 번호로 롤백하려면 어떻게 하나요?"*

**힌트:** `RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;`을 사용하세요.

In [ ]:
%sql
RESTORE TABLE banking_lab.silver.fact_transactions TO VERSION AS OF 1;

In [ ]:
%sql
-- 확인: 원래 금액이 복원되어야 합니다
SELECT transaction_id, amount, description
FROM banking_lab.silver.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007');
-- 예상: TXN-003 = 250.00 'Grocery store', TXN-007 = 1500.00 'Business expense'